# 🎥 Synthetic CCTV Supermarket Pipeline — Colab

End-to-end pipeline:

1. Install dependencies
2. Load the diffusion model
3. Generate the synthetic dataset (diffusion, CCTV-style prompts)
4. Augmentation (noise / blur / JPEG / perspective / fisheye / timestamp)
5. YOLOv8 person detection
6. Crop + letterbox (20% box expansion, 224×224, no stretching)
7. Train the multi-label classifier (10 sigmoid outputs, BCEWithLogitsLoss)
8. Inference demo

> **Runtime**: use a GPU runtime (Runtime → Change runtime type → T4/A100) if using the
> local Stable Diffusion backend. Not required for the `openai` or `mock` backends.
>
> **Demo vs full run**: `IMAGES_PER_LABEL` below defaults to a small demo value so the
> notebook finishes quickly. Set it to **100** (and `EPOCHS` to 20) for the full dataset.

## SECTION 1 — Install dependencies

In [ ]:
%pip install -q torch torchvision diffusers transformers accelerate safetensors \
    ultralytics "albumentations>=1.3.0,<1.5.0" opencv-python-headless pyyaml scikit-learn tqdm openai

import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

### Get the project code

Either clone your repository (set `REPO_URL`) **or** upload a zip of the `project/` folder when prompted.

In [ ]:
import os
from pathlib import Path

REPO_URL = ""  # e.g. "https://github.com/<you>/<repo>.git" — leave empty to upload a zip

if REPO_URL:
    !git clone $REPO_URL project_repo
    # If the repo root IS the project, adjust the path below accordingly.
    PROJECT_DIR = "project_repo/project" if Path("project_repo/project").exists() else "project_repo"
elif not Path("project").exists():
    from google.colab import files
    print("Upload a zip containing the project/ folder:")
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    !unzip -qo "$zip_name"
    PROJECT_DIR = "project"
else:
    PROJECT_DIR = "project"

%cd $PROJECT_DIR
print("Working directory:", os.getcwd())

In [ ]:
# Pipeline scale — demo defaults; set IMAGES_PER_LABEL = 100 and EPOCHS = 20 for the full run.
IMAGES_PER_LABEL = 8
EPOCHS = 5
BACKEND = "sd"   # "sd" = local Stable Diffusion (GPU) | "openai" = OpenAI Images API | "mock" = fast procedural placeholder (no GPU/API)

### Diffusion backend & prompt configuration

Pick the image-generation backend and, optionally, override the scene-level base/negative
prompt that gets prepended to every label-specific prompt. Per-label fragments (e.g. "right
hand inside pants pocket") and the diversity axes (lighting, camera angle, crowd, clothing,
layout, image-quality degradation) always stay the same — only the shared scene description
is editable here.

If `BACKEND = "openai"`, paste an OpenAI API key when prompted (input is hidden and never
written to disk or committed to the config file — only held in this session's memory).

In [ ]:
import os
import yaml

# --- Custom base/negative prompt (leave as-is to use the built-in defaults) ---
BASE_PROMPT = (
    "CCTV footage inside a supermarket, wide-angle security camera, "
    "grainy low resolution, realistic lighting, aisles with products, "
    "people shopping, surveillance camera perspective"
)
NEGATIVE_PROMPT = (
    "cartoon, illustration, anime, painting, drawing, 3d render, cgi, "
    "text overlay, watermark, logo, deformed hands, extra fingers, extra limbs, "
    "duplicate person, close-up portrait, bokeh, shallow depth of field, "
    "professional photography, studio lighting, high fashion"
)

# --- OpenAI Images API settings (only used when BACKEND == "openai") ---
OPENAI_MODEL = "gpt-image-1"
OPENAI_SIZE = "1024x1024"

if BACKEND == "openai":
    from getpass import getpass

    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")
    print("OPENAI_API_KEY set for this session.")
    print("Note: the OpenAI Images API bills per image — "
          f"{IMAGES_PER_LABEL} images/label x 10 labels = {IMAGES_PER_LABEL * 10} images per run.")

# Write an effective config on top of configs/diffusion.yaml with the choices
# above, so the subprocess-based pipeline stages (Section 3) pick them up.
# The API key itself is never written to this file — only the env var above.
diffusion_cfg = yaml.safe_load(open("configs/diffusion.yaml"))
diffusion_cfg.update(
    {
        "backend": BACKEND,
        "images_per_label": IMAGES_PER_LABEL,
        "base_prompt": BASE_PROMPT,
        "negative_prompt": NEGATIVE_PROMPT,
        "openai_model": OPENAI_MODEL,
        "openai_size": OPENAI_SIZE,
    }
)
COLAB_DIFFUSION_CONFIG = "configs/diffusion_colab.yaml"
yaml.safe_dump(diffusion_cfg, open(COLAB_DIFFUSION_CONFIG, "w"))
print(f"Wrote {COLAB_DIFFUSION_CONFIG} (backend={BACKEND})")

## SECTION 2 — Load diffusion model

Sanity-checks the prompt generator (using the `BASE_PROMPT`/`NEGATIVE_PROMPT` set above) with
a fast placeholder preview — see the note in the next cell for why it doesn't load the real
diffusion model here. The real model/backend chosen above (`sd`, `openai`, or `mock`) loads
once, in Section 3, via `configs/diffusion_colab.yaml`.

In [ ]:
import sys
sys.path.insert(0, ".")

from diffusion.prompt_generator import PromptGenerator
from labels import LABELS

prompt_gen = PromptGenerator(seed=42, base_prompt=BASE_PROMPT, negative_prompt=NEGATIVE_PROMPT)
sample = prompt_gen.generate("right_hand_in_pocket")
print("Prompt:\n", sample["prompt"])
print("\nGround-truth labels:", {k: v for k, v in sample["labels"].items() if v})

In [ ]:
import matplotlib.pyplot as plt

# This preview always uses the fast, no-GPU mock backend. Loading the real SD
# pipeline here AND again in Section 3 (a separate `!python` subprocess) would
# hold two full copies of the model in memory at once — the actual cause of
# a "RAM reached maximum" crash on backend="sd". The real model/backend loads
# exactly once, in Section 3, no matter which BACKEND you selected above.
from diffusion.local_sd_runner import MockDiffusionRunner

preview_runner = MockDiffusionRunner()
test_image = preview_runner.generate(sample["prompt"], sample["negative_prompt"], seed=42)
plt.figure(figsize=(6, 6)); plt.imshow(test_image); plt.axis("off")
plt.title("Prompt/label preview (mock backend — real generation happens in Section 3)"); plt.show()
del preview_runner

## SECTION 3 — Generate synthetic dataset

≥ `IMAGES_PER_LABEL` images for **each of the 10 labels**, with diversity across lighting,
camera angle, crowd density, clothing, store layout and degradation, using
`configs/diffusion_colab.yaml` (backend, prompts, and OpenAI settings from the cell above).
Full metadata (prompt, seed, backend, label vector) is logged to
`dataset/synthetic_annotations.json`. Resumable — if this cell crashes or is interrupted,
just re-run it; already-generated images are skipped.

> **If you hit an out-of-memory crash** (only relevant for `BACKEND = "sd"`):
> `Runtime → Restart session` first, then re-run from Section 1. The default local config
> (SD 1.5, 512×512, `low_vram: true`) already loads the fp16 weight variant with
> `enable_model_cpu_offload()`, which should fit comfortably in free-tier Colab. If you
> switched to SDXL, either use a High-RAM/A100 runtime or lower `IMAGES_PER_LABEL` first.
> The `openai` backend makes no local GPU/RAM demands at all — it calls the OpenAI Images API.

In [ ]:
!python main_pipeline.py --stage generate --diffusion-config configs/diffusion_colab.yaml

In [ ]:
import json, cv2, matplotlib.pyplot as plt

annotations = json.load(open("dataset/synthetic_annotations.json"))
print(f"{len(annotations)} raw images generated")

fig, axes = plt.subplots(2, 5, figsize=(18, 8))
step = max(1, len(annotations) // 10)
for ax, entry in zip(axes.flat, annotations[::step]):
    img = cv2.cvtColor(cv2.imread(entry["image_path"]), cv2.COLOR_BGR2RGB)
    ax.imshow(img); ax.set_title(entry["primary_label"], fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

## SECTION 4 — Run augmentation

Gaussian noise, motion blur, brightness/contrast, JPEG artifacts, perspective, crop jitter
(albumentations) + fisheye distortion and burned-in CCTV timestamp (custom).

In [ ]:
!python main_pipeline.py --stage augment

In [ ]:
aug = [e for e in json.load(open("dataset/augmented_annotations.json")) if e.get("source") == "augmented"]
print(f"{len(aug)} augmented variants")

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, entry in zip(axes.flat, aug[:: max(1, len(aug) // 5)]):
    img = cv2.cvtColor(cv2.imread(entry["image_path"]), cv2.COLOR_BGR2RGB)
    ax.imshow(img); ax.set_title(entry["primary_label"], fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

## SECTION 5 — YOLO person detection

Pretrained YOLOv8 (COCO class 0 = person). Demo on one generated frame.

In [ ]:
from yolo.detect_people import PersonDetector, draw_detections

detector = PersonDetector(weights="yolov8n.pt", conf=0.35)
frame = cv2.imread(annotations[0]["image_path"])
detections = detector.detect(frame)
print(f"{len(detections)} person(s):", [round(d['conf'], 2) for d in detections])

plt.figure(figsize=(8, 8))
plt.imshow(cv2.cvtColor(draw_detections(frame, detections), cv2.COLOR_BGR2RGB))
plt.axis("off"); plt.title("YOLOv8 person detection"); plt.show()

## SECTION 6 — Crop + letterbox pipeline

Per image: detect the primary person → expand the box by **20%** → crop → **letterbox** to
224×224 (aspect-preserving, no stretching). Writes the final training annotations to
`dataset/annotations.json`.

In [ ]:
!python main_pipeline.py --stage crop

In [ ]:
crops = json.load(open("dataset/annotations.json"))
n_fallback = sum(e["fallback_full_image"] for e in crops)
print(f"{len(crops)} crops ({n_fallback} full-image fallbacks where YOLO found no person)")

fig, axes = plt.subplots(1, 6, figsize=(16, 3))
for ax, entry in zip(axes.flat, crops[:: max(1, len(crops) // 6)]):
    img = cv2.cvtColor(cv2.imread(entry["image_path"]), cv2.COLOR_BGR2RGB)
    active = [l for l, v in entry["labels"].items() if v]
    ax.imshow(img); ax.set_title("\n".join(active), fontsize=7); ax.axis("off")
plt.tight_layout(); plt.show()

## SECTION 7 — Train classifier

ResNet18 backbone → 10 **sigmoid** outputs (multi-label — never softmax), trained with
`BCEWithLogitsLoss` (+ pos_weight for imbalance). Fixed seeds; per-label P/R/F1, macro-F1
and mAP logged every epoch; best checkpoint by macro-F1.

In [ ]:
!python main_pipeline.py --stage train --epochs $EPOCHS

In [ ]:
log = json.load(open("model/training_log.json"))
history = log["history"]
print(f"Best macro-F1: {log['best_macro_f1']:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs_x = [h["epoch"] for h in history]
axes[0].plot(epochs_x, [h["train_loss"] for h in history]); axes[0].set_title("Train loss"); axes[0].set_xlabel("epoch")
axes[1].plot(epochs_x, [h["macro_f1"] for h in history], label="macro F1")
axes[1].plot(epochs_x, [h["mAP"] or 0 for h in history], label="mAP")
axes[1].set_title("Validation metrics"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout(); plt.show()

best = max(history, key=lambda h: h["macro_f1"])
for label, m in best["per_label"].items():
    print(f"{label:<28} P={m['precision']:.3f} R={m['recall']:.3f} F1={m['f1']:.3f}")

## SECTION 8 — Run inference demo

Full-frame inference: YOLO detects people → each crop is letterboxed and classified →
per-person sigmoid probabilities for all 10 labels.

In [ ]:
from model.inference import InferencePipeline

pipeline = InferencePipeline(checkpoint_path="model/checkpoints/best.pt", threshold=0.5)

demo_frame = cv2.imread(annotations[3]["image_path"])
results = pipeline.predict_frame(demo_frame)
for r in results:
    print("box:", r["box"], "| active:", r["active_labels"])
    top = sorted(r["probabilities"].items(), key=lambda kv: -kv[1])[:5]
    for label, p in top:
        print(f"   {label:<28} {p:.3f}")

plt.figure(figsize=(9, 9))
plt.imshow(cv2.cvtColor(pipeline.annotate_frame(demo_frame, results), cv2.COLOR_BGR2RGB))
plt.axis("off"); plt.title("Inference demo"); plt.show()

---
### ✅ Outputs produced

| Artifact | Path |
|---|---|
| Synthetic dataset (≥100/label at full scale) | `dataset/synthetic_raw/` |
| Augmented dataset | `dataset/augmented/` |
| Cropped 224×224 person dataset | `dataset/cropped/` |
| Final annotations | `dataset/annotations.json` |
| Trained multi-label classifier | `model/checkpoints/best.pt` |
| Training log & metrics | `model/training_log.json` |

To persist results, zip and download or copy to Drive:
```python
!zip -qr outputs.zip dataset/annotations.json model/checkpoints model/training_log.json
from google.colab import files; files.download("outputs.zip")
```